# Bellabeat Data Analysis

## 01 — Data Exploration & Quality Assessment

## 1. Dataset Inventory

### 1.1 Source Files

In [ ]:
from pathlib import Path

cwd = Path.cwd()
project_root = cwd if (cwd / "data" / "raw").exists() else cwd.parent
raw_dir = project_root / "data" / "raw"

assert raw_dir.is_dir(), "Raw data directory not found."

total_files = 0

for folder in sorted(raw_dir.iterdir()):
    if folder.is_dir():
        csv_files = sorted(folder.glob("*.csv"))
        total_files += len(csv_files)

        print(f"\n{folder.name}: {len(csv_files)} CSV files")

        for file in csv_files:
            print(f"  - {file.name}")

print(f"\nTotal CSV files: {total_files}")



### 1.2 Dataset Overview


In [ ]:
import pandas as pd

# Select the dataset period
data_dir = raw_dir / "Fitabase Data 4.12.16-5.12.16"

# Load the primary datasets
activity_df = pd.read_csv(data_dir / "dailyActivity_merged.csv")
sleep_df = pd.read_csv(data_dir / "sleepDay_merged.csv")

# Display dataset dimensions
print("Activity shape:", activity_df.shape)
print("Sleep shape:", sleep_df.shape)

# Preview the datasets
print("\nDaily Activity:")
display(activity_df.head())

print("\nSleep Data:")
display(sleep_df.head())


### 1.3 Data Types

In [ ]:
print("Daily Activity - Data Types:")
activity_df.info()

print("\nSleep Data - Data Types:")
sleep_df.info()

### 1.4 User Coverage & Date Range

In [ ]:
# Count unique users
print("Activity unique users:", activity_df["Id"].nunique())
print("Sleep unique users:", sleep_df["Id"].nunique())

# Count users appearing in both datasets
common_users = set(activity_df["Id"]) & set(sleep_df["Id"])
print("Common users:", len(common_users))

# Parse dates without modifying the original DataFrames
activity_dates = pd.to_datetime(
    activity_df["ActivityDate"],
    format="%m/%d/%Y",
    errors="coerce"
)

sleep_dates = pd.to_datetime(
    sleep_df["SleepDay"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

# Display date ranges
print("\nActivity date range:")
print(activity_dates.min(), "to", activity_dates.max())

print("\nSleep date range:")
print(sleep_dates.min(), "to", sleep_dates.max())

# Check for invalid dates
print("\nInvalid activity dates:", activity_dates.isna().sum())
print("Invalid sleep dates:", sleep_dates.isna().sum())


### 1.5 Missing Values

In [6]:
# Check missing values in Daily Activity
activity_missing = pd.DataFrame({
    "Missing Count": activity_df.isna().sum(),
    "Missing Percentage": (activity_df.isna().mean() * 100).round(2)
})

print("Daily Activity - Missing Values:")
display(activity_missing)

# Check missing values in Sleep Data
sleep_missing = pd.DataFrame({
    "Missing Count": sleep_df.isna().sum(),
    "Missing Percentage": (sleep_df.isna().mean() * 100).round(2)
})

print("\nSleep Data - Missing Values:")
display(sleep_missing)

Daily Activity - Missing Values:


,Missing Count,Missing Percentage
Id,0,0.0
ActivityDate,0,0.0
TotalSteps,0,0.0
TotalDistance,0,0.0
TrackerDistance,0,0.0
LoggedActivitiesDistance,0,0.0
VeryActiveDistance,0,0.0
ModeratelyActiveDistance,0,0.0
LightActiveDistance,0,0.0
SedentaryActiveDistance,0,0.0



Sleep Data - Missing Values:


,Missing Count,Missing Percentage
Id,0,0.0
SleepDay,0,0.0
TotalSleepRecords,0,0.0
TotalMinutesAsleep,0,0.0
TotalTimeInBed,0,0.0


### 1.6 Duplicate Records

In [7]:
# Check exact duplicate rows
print("Daily Activity - Exact Duplicates:",
      activity_df.duplicated().sum())

print("Sleep Data - Exact Duplicates:",
      sleep_df.duplicated().sum())

# Check duplicate user-date combinations
print("\nDaily Activity - Duplicate User-Date Records:",
      activity_df.duplicated(subset=["Id", "ActivityDate"]).sum())

print("Sleep Data - Duplicate User-Date Records:",
      sleep_df.duplicated(subset=["Id", "SleepDay"]).sum())


Daily Activity - Exact Duplicates: 0
Sleep Data - Exact Duplicates: 3

Daily Activity - Duplicate User-Date Records: 0
Sleep Data - Duplicate User-Date Records: 3


### 1.7 Data Integrity & Quality Assessment

In [8]:
# Check negative values in activity measurements
activity_numeric = activity_df.select_dtypes(include="number").drop(columns=["Id"])

activity_negative = (activity_numeric < 0).sum()

print("Daily Activity - Negative Values:")
display(activity_negative.to_frame(name="Negative Count"))

# Check negative values in sleep measurements
sleep_numeric = sleep_df.select_dtypes(include="number").drop(columns=["Id"])

sleep_negative = (sleep_numeric < 0).sum()

print("\nSleep Data - Negative Values:")
display(sleep_negative.to_frame(name="Negative Count"))

# Check sleep duration consistency
invalid_sleep_duration = (
    sleep_df["TotalMinutesAsleep"] > sleep_df["TotalTimeInBed"]
).sum()

print("\nSleep duration exceeding time in bed:", invalid_sleep_duration)

# Check invalid sleep record counts
invalid_sleep_records = (
    sleep_df["TotalSleepRecords"] <= 0
).sum()

print("Non-positive sleep record counts:", invalid_sleep_records)

# Check activity minutes exceeding 24 hours
activity_minutes = (
    activity_df["VeryActiveMinutes"]
    + activity_df["FairlyActiveMinutes"]
    + activity_df["LightlyActiveMinutes"]
    + activity_df["SedentaryMinutes"]
)

print("Activity records exceeding 1440 minutes:",
      (activity_minutes > 1440).sum())


Daily Activity - Negative Values:


,Negative Count
TotalSteps,0
TotalDistance,0
TrackerDistance,0
LoggedActivitiesDistance,0
VeryActiveDistance,0
ModeratelyActiveDistance,0
LightActiveDistance,0
SedentaryActiveDistance,0
VeryActiveMinutes,0
FairlyActiveMinutes,0



Sleep Data - Negative Values:


,Negative Count
TotalSleepRecords,0
TotalMinutesAsleep,0
TotalTimeInBed,0



Sleep duration exceeding time in bed: 0
Non-positive sleep record counts: 0
Activity records exceeding 1440 minutes: 0
